# Questão 1 — EDA da tabela orders"

## Parte 1 Visão Geral

In [ ]:
# importação das biblioteca necessária
import duckdb as db

In [15]:
caminho_orders = '1-lh_nautical_csv/orders.csv'
db.sql(f"SELECT * FROM '{caminho_orders}' LIMIT 10")

┌───────┬──────────────┬───────────┬─────────────┬────────────────┬─────────────┬───────────┬──────────┬─────────────────┬──────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│  id   │ order_number │  channel  │ customer_id │ salesperson_id │ location_id │  status   │ subtotal │ discount_amount │  total   │      placed_at      │     created_at      │     updated_at      │
│ int64 │   varchar    │  varchar  │    int64    │     int64      │    int64    │  varchar  │  double  │     double      │  double  │      timestamp      │      timestamp      │      timestamp      │
├───────┼──────────────┼───────────┼─────────────┼────────────────┼─────────────┼───────────┼──────────┼─────────────────┼──────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│     1 │ SO-000001    │ ecommerce │        1136 │           NULL │           1 │ paid      │   323.34 │           35.57 │   287.77 │ 2022-09-06 05:37:37 │ 2022-09-06 05:37:37 │ 2022-09-06 05:37:37 │


Quantidade total de linhas

In [16]:
db.sql(f"SELECT COUNT(*) FROM '{caminho_orders}'")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        48998 │
└──────────────┘

Ou seja um total de 48998 linhas

Data Miníma e data Máxima

In [27]:
db.sql(f"SELECT  MIN(created_at) AS data_minima FROM '{caminho_orders}' ")


┌─────────────────────┐
│     data_minima     │
│      timestamp      │
├─────────────────────┤
│ 2020-01-01 01:19:28 │
└─────────────────────┘

In [28]:
db.sql(f"SELECT  MAX(created_at) AS data_maxima FROM '{caminho_orders}'")

┌─────────────────────┐
│     data_maxima     │
│      timestamp      │
├─────────────────────┤
│ 2026-12-31 23:43:09 │
└─────────────────────┘

Percebe-se um problema, existe uma data_máxima que está no futuro, ou seja, como estamos em agosto de 2026 parte dessas datas não seriam consideradas

## 2. Análise de valores

In [31]:
# Valor máximo
db.sql(f"SELECT MAX(total) as valor_maximo FROM '{caminho_orders}'")

┌──────────────┐
│ valor_maximo │
│    double    │
├──────────────┤
│    127262.02 │
└──────────────┘

In [32]:
# Valor minímo
db.sql(f"SELECT MIN(total) as valor_minimo FROM '{caminho_orders}'")

┌──────────────┐
│ valor_minimo │
│    double    │
├──────────────┤
│        32.62 │
└──────────────┘

In [33]:
# Valor medio
db.sql(f"SELECT AVG(total) as valor_medio FROM '{caminho_orders}'")

┌────────────────────┐
│    valor_medio     │
│       double       │
├────────────────────┤
│ 28704.992077227675 │
└────────────────────┘

# Possíveis outliers na coluna total
    O valor máximo é muito maior que a média por isso vou calcular a variância para prova que é um outlier
    Ou seja, existe uma dispersão de dados muito grande

## Parte 3 - Interpretação

In [46]:
# Valor do desvio padrão (raiz quadrada da variância) 
db.sql(f"SELECT STDDEV(total) as Desvio_Padrão FROM '{caminho_orders}'")

┌─────────────────┐
│  Desvio_Padrão  │
│     double      │
├─────────────────┤
│ 19425.636818482 │
└─────────────────┘

In [48]:
desvio_total = (127262.02 - 28704.99) / 19425.64
print(desvio_total)

5.0735538185614475


In [39]:
# Valores nulos
db.sql(f"SELECT COUNT(*)  as Nulos FROM '{caminho_orders}' WHERE total IS NULL")

┌───────┐
│ Nulos │
│ int64 │
├───────┤
│     0 │
└───────┘

In [ ]:
db.sql(f"SELECT COUNT(*)  as Nulos FROM '{caminho_orders}' WHERE customer_id IS NULL")

┌───────┐
│ Nulos │
│ int64 │
├───────┤
│     0 │
└───────┘

In [42]:
db.sql(f"SELECT COUNT(*)  as Nulos FROM '{caminho_orders}' WHERE created_at IS NULL")

┌───────┐
│ Nulos │
│ int64 │
├───────┤
│     0 │
└───────┘

Aqui é o seguinte vendas online não tem ajuda de vendedor portanto o indice nesse caso é nulo

In [ ]:
db.sql(f"SELECT COUNT(*)  as Nulos FROM '{caminho_orders}' WHERE salesperson_id IS NULL")

┌───────┐
│ Nulos │
│ int64 │
├───────┤
│ 24131 │
└───────┘

# 3. O Dataset está pronto para uso ou exige tratamento prévio?

Não porque apesar de não existir valores nulos em colunas importantes, há alguns problemas importantes:
1. O valor máximo está há 5,07 de desvios-padrão ,ou seja, os valores são muito destoantes, veja os valores
média = │ 28704.992077227675 │ valor_maximo = │    127262.02 │ Desvio_padrao = │ 19425.636818482 │
2. Apenas a coluna salesperson_id tem valores nulos mas pode significar apenas que tem vendas_online que não é
necessário um vendedor
3. Há datas futuras porque estamos em agosto de 2026 e existem datas até dezembro deste ano
Portanto concluimos que a tabela não está pronta para uso direto ela necessita de tratamento de dados
